In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [4]:
tornados_outage_2023 = pd.read_csv('../merged/tornados_outage_2023.csv', parse_dates=['DATE'])



In [5]:
tornados_outage_2023.head()

,Unnamed: 0,DATE,AVGDV_max,LLDV_max,MXDV_max,MXDV_HEIGHT_max,DEPTH_max,MAX_SHEAR_max,MAX_SHEAR_HEIGHT_max,location,county,state,Event Month,power_outage
0,0,2023-01-01,36,49,56,7,6.5,29,6.6,"(32.39305, -110.68147)",Pima County,Arizona,NaN,False
1,1,2023-01-01,52,96,96,2,5.9,86,1.8,"(32.18141, -110.52664)",Cochise County,Arizona,NaN,False
2,2,2023-01-01,31,50,50,4,9.1,17,4.2,"(34.34357, -117.82177)",San Bernardino County,California,NaN,False
3,3,2023-01-01,35,62,62,6,8.3,21,6.4,"(34.29869, -117.62836)",San Bernardino County,California,NaN,False
4,4,2023-01-01,39,52,52,4,5.0,24,2.6,"(35.0269, -118.24596)",Kern County,California,NaN,False


In [6]:
tornados_outage_2023['DATE'] = pd.to_datetime(tornados_outage_2023['DATE'])
tornados_outage_2023['Month'] = tornados_outage_2023['DATE'].dt.month

In [9]:
tornados_outage_2023.sample(5)

,Unnamed: 0,DATE,AVGDV_max,LLDV_max,MXDV_max,MXDV_HEIGHT_max,DEPTH_max,MAX_SHEAR_max,MAX_SHEAR_HEIGHT_max,location,county,state,Event Month,power_outage,Month
15740,15740,2023-05-28,48,61,87,9,13.0,35,9.0,"(26.4476, -79.70448)",Palm Beach County,Florida,NaN,False,5
12255,12255,2023-04-24,44,70,86,21,15.3,32,21.3,"(26.85117, -81.07203)",Glades County,Florida,NaN,False,4
25687,25687,2023-07-18,34,70,81,29,24.7,39,28.8,"(26.63871, -80.12101)",Palm Beach County,Florida,NaN,False,7
25763,25763,2023-07-18,54,94,94,1,11.0,76,1.3,"(40.85504, -74.1444)",Passaic County,New Jersey,NaN,False,7
5829,5829,2023-03-03,49,52,92,1,5.7,155,1.0,"(40.31369, -89.37858)",Tazewell County,Illinois,March,False,3


In [11]:
print(tornados_outage_2023['location'].dtype)

object


In [14]:
tornados_outage_2023[['Latitude', 'Longitude']] = tornados_outage_2023['location'].apply(lambda x: pd.Series(str(x).strip('()').split(',')))


In [16]:
tornados_outage_2023.sample(5)

,Unnamed: 0,DATE,AVGDV_max,LLDV_max,MXDV_max,MXDV_HEIGHT_max,DEPTH_max,MAX_SHEAR_max,MAX_SHEAR_HEIGHT_max,location,county,state,Event Month,power_outage,Month,Latitude,Longitude
33442,33442,2023-08-25,63,80,149,7,5.0,119,7.2,"(29.74381, -95.67256)",Fort Bend County,Texas,NaN,False,8,29.74381,-95.67256
20847,20847,2023-06-22,56,86,116,22,30.0,51,21.6,"(28.624956, -80.64447)",Brevard County,Florida,NaN,False,6,28.624956,-80.64447
30537,30537,2023-08-07,73,104,104,20,27.4,59,20.1,"(38.55156625, -85.84086625)",Clark County,Indiana,NaN,False,8,38.55156625,-85.84086625
14261,14261,2023-05-13,48,92,93,25,21.1,35,25.2,"(35.9697625, -97.5769625)",Logan County,Oklahoma,NaN,False,5,35.9697625,-97.5769625
7616,7616,2023-03-24,47,86,86,2,19.6,33,1.7,"(39.39163, -85.21854)",Ripley County,Indiana,NaN,False,3,39.39163,-85.21854


In [17]:
tornados_grouped_by = tornados_outage_2023.groupby(['power_outage'])
tornados_balanced = tornados_grouped_by.apply(lambda x: x.sample(tornados_grouped_by.size().min()).reset_index(drop=True))
tornados_balanced =tornados_balanced.droplevel(['power_outage'])

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
tornados_train, tornados_test = train_test_split(tornados_balanced.copy(),
                                              shuffle=True,
                                              random_state=123,
                                              test_size=.1,
                                              stratify=tornados_balanced.power_outage.values)

In [20]:
## import random forest classifier
from sklearn.ensemble import RandomForestClassifier

## import kfold
from sklearn.model_selection import StratifiedKFold

## import accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score



In [8]:
#def powerset(s):
#    power_set = [[]]
#    for x in s:
#        power_set += [s0+[x] for s0 in power_set]
#    return power_set[1:]

In [9]:
#potential_features = ['AVGDV_max', 'LLDV_max', 'MXDV_max', 'MXDV_HEIGHT_max', 'DEPTH_max',
#      'MAX_SHEAR_max', 'MAX_SHEAR_HEIGHT_max', 'state']

#all_models = ['baseline']
#all_models.extend(powerset(potential_features))

In [37]:
## this will isolate the feature columns
features = ['AVGDV_max', 'LLDV_max', 'MXDV_max', 'MXDV_HEIGHT_max', 'DEPTH_max',
       'MAX_SHEAR_max', 'MAX_SHEAR_HEIGHT_max', 'Month', 'Latitude', 'Longitude' ]

In [38]:

features

['AVGDV_max',
 'LLDV_max',
 'MXDV_max',
 'MXDV_HEIGHT_max',
 'DEPTH_max',
 'MAX_SHEAR_max',
 'MAX_SHEAR_HEIGHT_max',
 'Month',
 'Latitude',
 'Longitude']

In [39]:
## set the number of CV folds
n_splits = 5

## Make the kfold object
kfold = StratifiedKFold(n_splits, 
                        random_state=216, 
                        shuffle=True)

In [40]:
max_depths = range(1, 11)
n_trees = [100, 500]

rf_accs = np.zeros((n_splits, len(max_depths), len(n_trees)))



for i,(train_index, test_index) in enumerate(kfold.split(tornados_train, tornados_train.power_outage)):
    tornados_tt = tornados_train.iloc[train_index]
    tornados_ho = tornados_train.iloc[test_index]

    for j, max_depth in enumerate(max_depths):
        for k, n_estimators in enumerate(n_trees):
            print(i,j,k)
            rf = RandomForestClassifier(n_estimators = n_estimators,
                                           max_depth = max_depth,
                                           max_samples = 0.8,
                                           random_state = 216)
                                           
            rf.fit(tornados_tt[features], tornados_tt.power_outage)
            
            pred = rf.predict(tornados_ho[features])
            
            rf_accs[i,j,k] = accuracy_score(tornados_ho.power_outage,  pred)

0 0 0
0 0 1
0 1 0
0 1 1
0 2 0
0 2 1
0 3 0
0 3 1
0 4 0
0 4 1
0 5 0
0 5 1
0 6 0
0 6 1
0 7 0
0 7 1
0 8 0
0 8 1
0 9 0
0 9 1
1 0 0
1 0 1
1 1 0
1 1 1
1 2 0
1 2 1
1 3 0
1 3 1
1 4 0
1 4 1
1 5 0
1 5 1
1 6 0
1 6 1
1 7 0
1 7 1
1 8 0
1 8 1
1 9 0
1 9 1
2 0 0
2 0 1
2 1 0
2 1 1
2 2 0
2 2 1
2 3 0
2 3 1
2 4 0
2 4 1
2 5 0
2 5 1
2 6 0
2 6 1
2 7 0
2 7 1
2 8 0
2 8 1
2 9 0
2 9 1
3 0 0
3 0 1
3 1 0
3 1 1
3 2 0
3 2 1
3 3 0
3 3 1
3 4 0
3 4 1
3 5 0
3 5 1
3 6 0
3 6 1
3 7 0
3 7 1
3 8 0
3 8 1
3 9 0
3 9 1
4 0 0
4 0 1
4 1 0
4 1 1
4 2 0
4 2 1
4 3 0
4 3 1
4 4 0
4 4 1
4 5 0
4 5 1
4 6 0
4 6 1
4 7 0
4 7 1
4 8 0
4 8 1
4 9 0
4 9 1


In [41]:
max_depths = range(1, 11)
n_trees = [100, 500]

rf_accs = np.zeros((n_splits, len(max_depths), len(n_trees)))
rf_prec = np.zeros((n_splits, len(max_depths), len(n_trees)))
rf_recall = np.zeros((n_splits, len(max_depths), len(n_trees)))



for i,(train_index, test_index) in enumerate(kfold.split(tornados_train, tornados_train.power_outage)):
    tornados_tt = tornados_train.iloc[train_index]
    tornados_ho = tornados_train.iloc[test_index]

    for j, max_depth in enumerate(max_depths):
        for k, n_estimators in enumerate(n_trees):
            print(i,j,k)
            rf = RandomForestClassifier(n_estimators = n_estimators,
                                           max_depth = max_depth,
                                           max_samples = 0.8,
                                           random_state = 216)
                                           
            rf.fit(tornados_tt[features], tornados_tt.power_outage)
            
            pred = rf.predict(tornados_ho[features])
            
            rf_accs[i,j,k] = accuracy_score(tornados_ho.power_outage,  pred)
            rf_prec[i,j,k]= precision_score(tornados_ho.power_outage,  pred)
            rf_recall[i,j,k]= recall_score(tornados_ho.power_outage,  pred)

0 0 0
0 0 1
0 1 0
0 1 1
0 2 0
0 2 1
0 3 0
0 3 1
0 4 0
0 4 1
0 5 0
0 5 1
0 6 0
0 6 1
0 7 0
0 7 1
0 8 0
0 8 1
0 9 0
0 9 1
1 0 0
1 0 1
1 1 0
1 1 1
1 2 0
1 2 1
1 3 0
1 3 1
1 4 0
1 4 1
1 5 0
1 5 1
1 6 0
1 6 1
1 7 0
1 7 1
1 8 0
1 8 1
1 9 0
1 9 1
2 0 0
2 0 1
2 1 0
2 1 1
2 2 0
2 2 1
2 3 0
2 3 1
2 4 0
2 4 1
2 5 0
2 5 1
2 6 0
2 6 1
2 7 0
2 7 1
2 8 0
2 8 1
2 9 0
2 9 1
3 0 0
3 0 1
3 1 0
3 1 1
3 2 0
3 2 1
3 3 0
3 3 1
3 4 0
3 4 1
3 5 0
3 5 1
3 6 0
3 6 1
3 7 0
3 7 1
3 8 0
3 8 1
3 9 0
3 9 1
4 0 0
4 0 1
4 1 0
4 1 1
4 2 0
4 2 1
4 3 0
4 3 1
4 4 0
4 4 1
4 5 0
4 5 1
4 6 0
4 6 1
4 7 0
4 7 1
4 8 0
4 8 1
4 9 0
4 9 1


We will do `GridSearchCV

In [42]:
max_index_prec = np.unravel_index(np.argmax(np.mean(rf_prec, axis=0)), 
                                       np.mean(rf_prec, axis=0).shape)


print(max_depths[max_index_prec[0]],n_trees[max_index_prec[1]])

10 500


In [43]:
max_index_recall = np.unravel_index(np.argmax(np.mean(rf_recall, axis=0)), 
                                       np.mean(rf_recall, axis=0).shape)


print(max_depths[max_index_recall[0]],n_trees[max_index_recall[1]])

5 500


In [44]:
np.mean(rf_recall, axis=0)

array([[0.97699156, 0.97699156],
       [0.97699156, 0.97469271],
       [0.97814763, 0.97929706],
       [0.98505083, 0.98620025],
       [0.98620025, 0.9884991 ],
       [0.9884991 , 0.98734968],
       [0.9884991 , 0.9884991 ],
       [0.9839014 , 0.98505083],
       [0.98389476, 0.98504418],
       [0.98158926, 0.98388811]])

In [45]:
np.mean(rf_prec, axis=0)

array([[0.77237277, 0.77463217],
       [0.79432878, 0.79459305],
       [0.795729  , 0.79672116],
       [0.79769976, 0.79717686],
       [0.80019443, 0.79974963],
       [0.80217628, 0.8011129 ],
       [0.80430177, 0.80511614],
       [0.81177563, 0.80958737],
       [0.8140156 , 0.81666212],
       [0.82229355, 0.82343625]])

In [46]:
max_index = np.unravel_index(np.argmax(np.mean(rf_accs, axis=0)), 
                                       np.mean(rf_accs, axis=0).shape)


print(max_depths[max_index[0]],n_trees[max_index[1]])

10 500


In [47]:
## first import GridSearchCV
from sklearn.model_selection import GridSearchCV

In [48]:
grid_cv = GridSearchCV(RandomForestClassifier(), # first put the model object here
                          param_grid = {'max_depth':max_depths, # place the grid values for max_depth and
                                        'n_estimators':n_trees}, # and n_estimators here
                          scoring = 'accuracy', # put the metric we are trying to optimize here as a string, "accuracy"
                          cv = 5) # put the number of cv splits here

## you fit it just like a model
grid_cv.fit(tornados_train[features], tornados_train.power_outage)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'max_depth': range(1, 11), 'n_estimators': [100, 500]},
             scoring='accuracy')

In [49]:
## You can find the hyperparameter grid point that
## gave the best performance like so
## .best_params_
grid_cv.best_params_

{'max_depth': 10, 'n_estimators': 500}

In [50]:
## You can find the best score like so
## .best_score_
grid_cv.best_score_

0.8934876941932492

In [51]:
## Calling best_estimator_ returns the model with the 
## best avg cv performance after it has been refit on the
## entire data set
grid_cv.best_estimator_

RandomForestClassifier(max_depth=10, n_estimators=500)

In [52]:
grid_cv.best_estimator_.predict(tornados_train[features])

array([ True, False, False, ...,  True, False, False])

In [53]:
## You can get all of the results with cv_results_
grid_cv.cv_results_

{'mean_fit_time': array([0.10704894, 0.52353354, 0.13675036, 0.66345119, 0.16037774,
        0.78133426, 0.18242869, 0.92910013, 0.20235314, 1.01588802,
        0.22687621, 1.12942443, 0.2509151 , 1.23391461, 0.27664185,
        1.35341287, 0.29024386, 1.43012896, 0.30556121, 1.51949568]),
 'std_fit_time': array([0.00975923, 0.0237532 , 0.00526362, 0.00725859, 0.00167259,
        0.0143517 , 0.00354084, 0.02557697, 0.00212328, 0.00824126,
        0.00363039, 0.00888483, 0.00686299, 0.003464  , 0.00512413,
        0.03027021, 0.00652465, 0.01523652, 0.00643568, 0.02233033]),
 'mean_score_time': array([0.00864539, 0.03277702, 0.00816855, 0.0337779 , 0.00822644,
        0.0363719 , 0.00835629, 0.0365098 , 0.008566  , 0.03833847,
        0.00902286, 0.03744888, 0.00873189, 0.03696561, 0.0087379 ,
        0.0386395 , 0.0098012 , 0.04089069, 0.00981779, 0.04002733]),
 'std_score_time': array([0.00051905, 0.00133889, 0.00089355, 0.00081766, 0.00041204,
        0.00349743, 0.00043031, 0.001590

Using either the `best_estimator_` fitted model or a refitted model according to our results from the `for` loop cross-validation we will find the feature importance scores. 

In [54]:
pd.DataFrame({'feature_importance_score':grid_cv.best_estimator_.feature_importances_},
                 index=features).sort_values('feature_importance_score',
                                                ascending=False)

,feature_importance_score
Month,0.544687
Longitude,0.156065
Latitude,0.117105
DEPTH_max,0.033234
MXDV_max,0.030708
MAX_SHEAR_max,0.029918
LLDV_max,0.025716
AVGDV_max,0.025106
MAX_SHEAR_HEIGHT_max,0.022798
MXDV_HEIGHT_max,0.014664


Bagging

In [55]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate

In [56]:
pipe = Pipeline([('scale', StandardScaler()),('knn',KNeighborsClassifier())])
bag_pipe = BaggingClassifier(pipe, bootstrap = True, max_samples = 0.90)
bag_cv = GridSearchCV(bag_pipe, 
                          param_grid = {'estimator__knn__n_neighbors':[1,2,3], 
                                        'n_estimators':np.arange(1,100,10)}, 
                          scoring = 'accuracy', 
                          cv = 5)
bag_cv.fit(tornados_train[features], tornados_train.power_outage)

GridSearchCV(cv=5,
             estimator=BaggingClassifier(estimator=Pipeline(steps=[('scale',
                                                                    StandardScaler()),
                                                                   ('knn',
                                                                    KNeighborsClassifier())]),
                                         max_samples=0.9),
             param_grid={'estimator__knn__n_neighbors': [1, 2, 3],
                         'n_estimators': array([ 1, 11, 21, 31, 41, 51, 61, 71, 81, 91])},
             scoring='accuracy')

In [57]:
print(f"The best mean cv accuracy of {bag_cv.best_score_:.3f} was achieved using k = {bag_cv.best_estimator_.estimator['knn'].n_neighbors} and {bag_cv.best_estimator_.n_estimators} estimators")

The best mean cv accuracy of 0.833 was achieved using k = 3 and 71 estimators


In [58]:
single_pipe = Pipeline([('scale', StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=3))])
single_cv = cross_validate(single_pipe, tornados_train[features], tornados_train.power_outage, cv = 5, scoring = 'accuracy')

In [59]:
print(f"The mean cv accuracy of a single kNN model with k=3 is {single_cv['test_score'].mean():.3f}")

The mean cv accuracy of a single kNN model with k=3 is 0.822


Model with `GridSearchCV`

In [60]:
model = grid_cv.best_estimator_

model.fit(tornados_train[features], tornados_train.power_outage)

RandomForestClassifier(max_depth=10, n_estimators=500)

In [61]:
accuracy_score(model.predict(tornados_train[features]), tornados_train.power_outage)

0.9360967184801382

In [62]:
accuracy_score(model.predict(tornados_test[features]), tornados_test.power_outage)

0.8704663212435233

In [63]:
precision_score(model.predict(tornados_test[features]), tornados_test.power_outage)

0.9895833333333334

In [64]:
recall_score(model.predict(tornados_test[features]), tornados_test.power_outage)

0.7983193277310925